# Project A - main comparison

Runs compaction, the sleep consolidation loop for every method, and all evaluation arms.
Resumable: re-running picks up from the last checkpointed sleep phase.

**Run `a0_precondition.ipynb` first.** If no compactor beat the positional control there,
ours and uniform replay sample from the same distribution and this comparison is dead by
construction.

Compaction runs under Qwen2.5-1.5B (the only configuration with signal); consolidation
targets Qwen2.5-0.5B. The event log is plain text, so the two stages decouple cleanly.
Budget ~4 GPU-hours per session.

Set `DATASET` in the next cell. `hotpotqa` is the primary set — natural prose, no marker
phrase, positional control at chance. `synthetic` is for the compaction-ratio sweep and
for debugging.

In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'   # cleared only if you prefer a Kaggle Dataset
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

if not looks_like_source(REPO):
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)
    if not looks_like_source(REPO) and GIT_URL:
        r = subprocess.run(['git', 'clone', GIT_URL, REPO], capture_output=True, text=True)
        print(r.stdout, r.stderr)
    assert looks_like_source(REPO), (
        'No source found. Either (a) run scripts/package_source.py locally, upload the zip '
        'as a Kaggle Dataset, and attach it via Add Input, or (b) set GIT_URL above. '
        f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])
exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

DATASET = 'hotpotqa'
HOTPOT = DATASET == 'hotpotqa'
CFG = 'configs/kaggle_hotpotqa.yaml' if HOTPOT else 'configs/kaggle.yaml'
RUNS = '/kaggle/working/artifacts/runs_hotpot' if HOTPOT else '/kaggle/working/artifacts/runs'
CASC = f'{RUNS}/cascading'
N_TRAIN, N_EVAL = 32, 48

# Qwen2.5-0.5B is the better HotpotQA compactor (+4.77 vs +3.75 sigma) and runs
# 2.4x faster, so compaction and consolidation share one backbone here.
COMPACTOR = 'Qwen/Qwen2.5-0.5B-Instruct'
TARGET = 'Qwen/Qwen2.5-0.5B-Instruct'
print(f'{DATASET}: compactor {COMPACTOR.split("/")[-1]}, target {TARGET.split("/")[-1]}')
print(f'{N_TRAIN} train / {N_EVAL} eval trajectories')


## Restore previous session

Attach the Kaggle Dataset holding `runs.zip` from the last session, then restore. Skips
cleanly on a first run.

In [ ]:
ARCHIVE = '/kaggle/input/myrios-runs/runs.zip'
run(f"python scripts/kaggle_sync.py restore --archive {ARCHIVE} --run-root {RUNS}")


In [ ]:
if HOTPOT:
    run(f"python data/load_hotpotqa.py --n-train {N_TRAIN} --n-eval {N_EVAL} --per-trajectory 4 --early-frac 1.0")
else:
    run(f"python data/generate_synthetic.py --n-train {N_TRAIN} --n-eval {N_EVAL} --n-turns 120")


In [ ]:
run(f"python scripts/preflight.py --config {CFG}")


## Compaction - the free labels

Check the span report before continuing. Zero fallbacks, zero empty keeps, lift above the
positional control.

In [ ]:
run(f"python baselines/cascading.py --config {CFG} --split both --out {CASC} --set model.base={COMPACTOR}")
run(f"python eval/span_report.py --events {CASC}/train_events.jsonl --show 6 --out {CASC}/train_span_report.json")


In [ ]:
run(f"python baselines/full_context.py --config {CFG} --split eval --mode full")
run(f"python baselines/full_context.py --config {CFG} --split eval --mode none")
run(f"python baselines/reflection.py --config {CFG} --events {CASC}/train_events.jsonl --out {CASC}/train_reflections.jsonl --set model.base={COMPACTOR}")


## Sleep consolidation

`--resume` restores the adapter, the event cursor and the reservoir buffer, so an
interrupted session continues rather than restarting. Run ours and uniform first: if the
gap between them is invisible, the remaining arms are decoration.

In [ ]:
def sleep_run(method, tag, extra=''):
    run_dir = f'{RUNS}/sleep_{tag}'
    refl = f'--reflections {CASC}/train_reflections.jsonl' if method == 'reflection' else ''
    run(f"python sleep/loop.py --config {CFG} --method {method} --events {CASC}/train_events.jsonl --val-events {CASC}/eval_events.jsonl --run-dir {run_dir} --resume --val-limit 64 {refl} {extra} --set model.base={TARGET}")
    run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs.zip --prune")
    return run_dir

# ours vs uniform first: if that gap is invisible the remaining arms are decoration.
sleep_run('compaction', 'compaction')
sleep_run('uniform', 'uniform')


In [ ]:
sleep_run('reflection', 'reflection')
sleep_run('compaction', 'compaction+mask', '--mask-head')

## Evaluation

Every adapter arm sees the same post-compaction context, so the adapter is the only
difference between them.

In [ ]:
REPORT = f'{RUNS}/report'
ARMS = [
    ('floor', f'{RUNS}/none_context/eval_contexts.jsonl', None),
    ('cascading', f'{CASC}/eval_contexts.jsonl', None),
    ('full', f'{RUNS}/full_context/eval_contexts.jsonl', None),
    ('ours', f'{CASC}/eval_contexts.jsonl', f'{RUNS}/sleep_compaction/latest/adapter'),
    ('uniform', f'{CASC}/eval_contexts.jsonl', f'{RUNS}/sleep_uniform/latest/adapter'),
    ('reflection', f'{CASC}/eval_contexts.jsonl', f'{RUNS}/sleep_reflection/latest/adapter'),
    ('ours+mask', f'{CASC}/eval_contexts.jsonl', f'{RUNS}/sleep_compaction+mask/latest/adapter'),
]
for label, ctx, adapter in ARMS:
    a = f'--adapter {adapter}' if adapter else ''
    run(f"python eval/retention.py --config {CFG} --contexts {ctx} --label {label} --out {REPORT}/{label} {a} --set model.base={TARGET}")


In [ ]:
run(f"python eval/report.py --report-dir {REPORT} --run-root {RUNS} --out docs/results.md")
from IPython.display import Image, display
display(Image(f'{REPORT}/figures/headline_retention.png'))
display(Image(f'{REPORT}/figures/ce_curves.png'))

In [ ]:
run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs.zip --prune")
print('Update the Kaggle Dataset with /kaggle/working/runs.zip before the session ends.')